# 01 - Exploratory Data Analysis (EDA)

**Goal:** understand the energy data *before* modelling.

Run `python main.py --step generate` first so `data/raw/energy_consumption.csv` exists.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))   # lets the notebook import src/

import pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import load_raw_data, summarize
from src.preprocessing import clean_data

raw = load_raw_data()
raw.head()

## 1. Data quality check
How many missing values, duplicates and suspicious readings are there?

In [ ]:
summarize(raw)

In [ ]:
clean, report = clean_data(raw)
report

## 2. Overall trend and seasonality

In [ ]:
clean['energy_kwh'].resample('D').sum().plot(figsize=(12,4), title='Daily energy (kWh)');

## 3. Daily profile: weekday vs weekend

In [ ]:
d = clean.copy()
d['hour'] = d.index.hour
d['weekend'] = d.index.dayofweek >= 5
d.pivot_table(index='hour', columns='weekend', values='energy_kwh').plot(figsize=(10,4), title='Average load by hour');

## 4. Weather effect

In [ ]:
clean.plot.scatter(x='temperature_c', y='energy_kwh', s=2, alpha=0.3, figsize=(7,5));
clean[['temperature_c','energy_kwh']].corr()

## Key observations (write your own here!)
1. Clear **daily cycle**: morning ramp, office plateau, evening peak around 19:00-20:00.
2. **Weekends and holidays** use ~18-22% less energy.
3. **Temperature above ~24 °C** drives consumption up (air-conditioning).
4. A slow **upward trend** (city growth) exists across the two years.

These observations justify the features in `src/features.py`.